In [1]:
#!/usr/bin/env python3
"""
RSA Lag Analysis — Hippocampus Only (LHP / RHP), Band × Pair (Long Format)
---------------------------------------------------------------------------
*** FRACTAL (frac) variant — uses ret_frac_f* columns ***

Runs the RSA lag pipeline for:
  Regions : LHP, RHP  (hippocampus only)
  Bands   : theta  (3–7 Hz,   freq idx 0–3)
            alpha  (7–12 Hz,  freq idx 4–6)
            beta   (12–40 Hz, freq idx 7–11)
            gamma  (40–128 Hz,freq idx 12–17)

For every trial, for every pair of clean recalls (i, j) with i < j,
for every band:
  - Slice the 18-freq fractal IRASA vector to the band's freq indices
    → gives a (N_channels × N_band_freqs) sub-matrix per recall
  - Flatten each sub-matrix to a 1-D vector
  - RSA_r = Pearson r between the two flattened vectors

Output is LONG format: one row per pair × band.

Output columns:
  subject, session, experiment, trial,
  region, hemisphere,
  band, band_freq_indices,
  output_pos_i, output_pos_j, output_lag,
  word_i, word_j,
  serial_pos_i, serial_pos_j, T_lag, SP_lag,
  RSA_r, n_channels, semantic_sim

Per-region × per-band CSVs:
  ./rsa_lag_hc_bands_frac/ALL_SUBJECTS_{exp}_{region}_{band}_rsa_lag_frac.csv

Per-region all-bands CSV:
  ./rsa_lag_hc_bands_frac/ALL_SUBJECTS_{exp}_{region}_allbands_rsa_lag_frac.csv

Combined (both HC regions × all bands):
  ./rsa_lag_hc_bands_frac/ALL_SUBJECTS_{exp}_hc_allbands_rsa_lag_frac.csv
"""

import warnings
import traceback
from pathlib import Path
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from scipy.spatial.distance import euclidean

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

EXPERIMENTS = ['DBOY1', 'EFRCourierReadOnly', 'EFRCourierOpenLoop']

INPUT_DIRS = {
    'DBOY1':               Path('./subject_results_DBOY1'),
    'EFRCourierReadOnly':  Path('./subject_results_EFRCourierReadOnly'),
    'EFRCourierOpenLoop':  Path('./subject_results_EFRCourierOpenLoop'),
}

OUTPUT_DIR = Path('./rsa_lag_hc_bands_frac')   # ← separate output folder
OUTPUT_DIR.mkdir(exist_ok=True)

# Hippocampus only
REGIONS    = ['LHP', 'RHP']
HEMISPHERE = {'LHP': 'left', 'RHP': 'right'}

# 18 log-spaced frequencies from 3–128 Hz:
# idx:  0     1     2     3     4     5      6      7      8
# Hz:  3.00  3.74  4.67  5.82  7.26  9.05  11.28  14.07  17.55
# idx:  9     10    11    12    13    14    15     16     17
# Hz: 21.88 27.29 34.03 42.44 52.92 66.00 82.31 102.64 128.00
N_FREQS = 18

BANDS = {
    'theta': list(range(0, 4)),   # 3.00 – 5.82 Hz
    'alpha': list(range(4, 7)),   # 7.26 – 11.28 Hz
    'beta':  list(range(7, 12)),  # 14.07 – 34.03 Hz
    'gamma': list(range(12, 18)), # 42.44 – 128.00 Hz
}
BAND_ORDER = ['theta', 'alpha', 'beta', 'gamma']

RET_FRAC_COLS = [f'ret_frac_f{i:02d}' for i in range(N_FREQS)]   # ← frac

# ---- Word2Vec ---------------------------------------------------------------
WORD2VEC_PATH = Path('/home1/noaherz/word2vec/GoogleNews-vectors-negative300.bin.gz')

OUTPUT_COLS = [
    'subject', 'session', 'experiment', 'trial',
    'region', 'hemisphere',
    'band', 'band_freq_indices',
    'output_pos_i', 'output_pos_j', 'output_lag',
    'word_i', 'word_j',
    'serial_pos_i', 'serial_pos_j', 'T_lag', 'SP_lag',
    'RSA_r', 'n_channels', 'semantic_sim',
]

# ============================================================================
# WORD2VEC LOADER + SIMILARITY
# ============================================================================

def load_word2vec(path: Path):
    if path is None or not path.exists():
        print(f"  ⚠  Word2Vec model not found at {path}. semantic_sim will be NaN.")
        return None
    try:
        import gensim.models as gensim_models
        print(f"  Loading Word2Vec from {path} …")
        model = gensim_models.KeyedVectors.load_word2vec_format(str(path), binary=True)
        try:
            vs = len(model)
        except TypeError:
            vs = len(model.vocab)
        print(f"  ✓ Word2Vec loaded — vocab: {vs:,}")
        return model
    except Exception as e:
        print(f"  ✗ Word2Vec load failed: {e}. semantic_sim will be NaN.")
        return None


def case_insensitive_similarity(word1: str, word2: str, model) -> Optional[float]:
    cases = [
        (word1.lower(), word2.lower()),
        (word1.lower(), word2.upper()),
        (word1.upper(), word2.lower()),
        (word1.upper(), word2.upper()),
    ]
    sims = []
    for w1, w2 in cases:
        try:
            sims.append(model.similarity(w1, w2))
        except KeyError:
            continue
    return float(max(sims)) if sims else None


def build_similarity_cache(words: set, model) -> dict:
    if model is None:
        return {}
    unique_words = sorted(w for w in words if isinstance(w, str))
    n = len(unique_words)
    print(f"    Building semsim cache: {n} words → {n*(n-1)//2} pairs …")
    cache = {}
    for i, w1 in enumerate(unique_words):
        for w2 in unique_words[i:]:
            key = frozenset({w1, w2})
            if key not in cache:
                sim = case_insensitive_similarity(w1, w2, model)
                cache[key] = sim if sim is not None else np.nan
    return cache


# ============================================================================
# VECTOR + STATISTICS HELPERS
# ============================================================================

def build_band_vector(recall_rows: pd.DataFrame,
                      channel_index,
                      band_freq_indices: list) -> np.ndarray:
    """
    Build a 1-D retrieval vector for one recall event and one band:
      1. Align rows to channel_index       → (N_ch, 18) matrix
      2. Slice to band_freq_indices        → (N_ch, N_band_freqs) sub-matrix
      3. Flatten                           → (N_ch * N_band_freqs,) vector

    Uses fractal (ret_frac_f*) columns.
    Missing channels are NaN (handled pairwise in safe_pearsonr).
    """
    ch_df = (
        recall_rows
        .drop_duplicates(subset='channel_idx')
        .set_index('channel_idx')
        .reindex(channel_index)
    )
    band_cols = [RET_FRAC_COLS[i] for i in band_freq_indices]   # ← frac
    mat = ch_df[band_cols].values   # (N_ch, N_band_freqs)
    return mat.flatten()


def safe_pearsonr(v1: np.ndarray, v2: np.ndarray) -> float:
    if len(v1) != len(v2):
        return np.nan
    mask = np.isfinite(v1) & np.isfinite(v2)
    if mask.sum() < 3:
        return np.nan
    r, _ = pearsonr(v1[mask], v2[mask])
    return float(r)


def safe_euclidean(x1, z1, x2, z2) -> float:
    if any(not np.isfinite(v) for v in (x1, z1, x2, z2)):
        return np.nan
    return float(euclidean([x1, z1], [x2, z2]))


def extract_scalar(series: pd.Series, field: str, context: str):
    unique_vals = series.dropna().unique()
    if len(unique_vals) > 1:
        warnings.warn(
            f"[{context}] '{field}' has {len(unique_vals)} distinct values "
            f"({unique_vals[:3]}…). Taking first.")
    return series.iloc[0]


# ============================================================================
# TRIAL-LEVEL PROCESSING
# ============================================================================

def process_trial_region(trial_df: pd.DataFrame,
                         region: str,
                         sim_cache: dict) -> List[Dict]:
    """
    For one (subject, session, trial, region):
      - Enumerate all valid recall pairs (i, j), i < j
      - For each pair × each band: compute RSA_r on the band-sliced frac vector
      - Returns list of dicts in OUTPUT_COLS order (long format)
    """
    rows = []

    output_positions = sorted(
        trial_df['recall_output_position'].unique(),
        key=lambda x: int(x),
    )
    if len(output_positions) < 2:
        return rows

    channel_index = sorted(trial_df['channel_idx'].unique(), key=int)
    sample_row    = trial_df.iloc[0]

    # ------------------------------------------------------------------
    # Pre-compute per-output-position metadata + per-band vectors
    # ------------------------------------------------------------------
    pos_data: Dict[int, Dict] = {}

    for op in output_positions:
        op_rows = trial_df[trial_df['recall_output_position'] == op]
        ctx = (f"subj={sample_row['subject']} sess={sample_row['session']} "
               f"trial={sample_row['trial']} region={region} op={op}")

        word       = extract_scalar(op_rows['recalled_word'],   'recalled_word',   ctx)
        serial_pos = extract_scalar(op_rows['serial_position'], 'serial_position', ctx)
        store_x    = extract_scalar(op_rows['store_x'],         'store_x',         ctx)
        store_z    = extract_scalar(op_rows['store_z'],         'store_z',         ctx)
        n_ch       = op_rows['channel_idx'].nunique()

        # Build one vector per band (sliced, not averaged)
        band_vectors = {
            band_name: build_band_vector(op_rows, channel_index, freq_idx)
            for band_name, freq_idx in BANDS.items()
        }

        pos_data[op] = {
            'band_vectors': band_vectors,
            'word':         word,
            'serial_pos':   serial_pos,
            'store_x':      store_x,
            'store_z':      store_z,
            'n_channels':   n_ch,
        }

    subject    = sample_row['subject']
    session    = sample_row['session']
    experiment = sample_row['experiment']
    trial      = sample_row['trial']

    # ------------------------------------------------------------------
    # Enumerate pairs × bands
    # ------------------------------------------------------------------
    for idx_i, op_i in enumerate(output_positions):
        for op_j in output_positions[idx_i + 1:]:
            d_i = pos_data[op_i]
            d_j = pos_data[op_j]

            output_lag = int(op_j) - int(op_i)
            T_lag      = abs(int(d_i['serial_pos']) - int(d_j['serial_pos']))
            SP_lag     = safe_euclidean(d_i['store_x'], d_i['store_z'],
                                        d_j['store_x'], d_j['store_z'])
            n_channels = min(d_i['n_channels'], d_j['n_channels'])

            # Semantic similarity: same for all bands — compute once per pair
            w_i = str(d_i['word']).lower() if pd.notna(d_i['word']) else None
            w_j = str(d_j['word']).lower() if pd.notna(d_j['word']) else None
            sem_sim = (
                sim_cache.get(frozenset({w_i, w_j}), np.nan)
                if (w_i and w_j and sim_cache)
                else np.nan
            )

            # One row per band
            for band_name in BAND_ORDER:
                freq_idx = BANDS[band_name]
                v_i      = d_i['band_vectors'][band_name]
                v_j      = d_j['band_vectors'][band_name]
                rsa_r    = safe_pearsonr(v_i, v_j)

                rows.append({
                    'subject':           subject,
                    'session':           session,
                    'experiment':        experiment,
                    'trial':             trial,
                    'region':            region,
                    'hemisphere':        HEMISPHERE[region],
                    'band':              band_name,
                    'band_freq_indices': str(freq_idx),
                    'output_pos_i':      op_i,
                    'output_pos_j':      op_j,
                    'output_lag':        output_lag,
                    'word_i':            d_i['word'],
                    'word_j':            d_j['word'],
                    'serial_pos_i':      d_i['serial_pos'],
                    'serial_pos_j':      d_j['serial_pos'],
                    'T_lag':             T_lag,
                    'SP_lag':            SP_lag,
                    'RSA_r':             rsa_r,
                    'n_channels':        n_channels,
                    'semantic_sim':      sem_sim,
                })

    return rows


# ============================================================================
# PER-EXPERIMENT RUNNER
# ============================================================================

def run_experiment(exp: str, w2v_model):
    print(f"\n{'='*70}")
    print(f"EXPERIMENT: {exp}")
    print(f"{'='*70}")

    input_dir  = INPUT_DIRS.get(exp)
    if input_dir is None:
        print(f"  ✗ No INPUT_DIR configured for '{exp}'.")
        return

    input_path = input_dir / f"ALL_SUBJECTS_{exp}_irasa_channel_wide.csv"
    if not input_path.exists():
        print(f"  ✗ Master CSV not found: {input_path}")
        return

    print(f"  Loading {input_path} …")
    df = pd.read_csv(input_path)
    print(f"  Loaded {len(df):,} rows | "
          f"{df['subject'].nunique()} subjects | "
          f"{df['session'].nunique()} sessions")

    # Filter to HC only upfront
    df = df[df['region'].isin(REGIONS)].copy()
    print(f"  After HC filter: {len(df):,} rows")
    if df.empty:
        print("  ✗ No hippocampal data found — check 'region' column values.")
        return

    df['channel_idx'] = df['channel_idx'].astype(int)

    # Build Word2Vec cache once per experiment (all HC words)
    all_words_lower = set(
        df['recalled_word'].dropna().astype(str).str.lower().unique()
    )
    sim_cache = build_similarity_cache(all_words_lower, w2v_model)

    all_region_dfs = []

    for region in REGIONS:
        print(f"\n  {'─'*60}")
        print(f"  Region: {region}")

        region_df = df[df['region'] == region].copy()
        if region_df.empty:
            print(f"  ✗ No rows for region {region} — skipping")
            continue

        print(f"  Rows: {len(region_df):,}")

        all_rows = []
        groups   = region_df.groupby(['subject', 'session', 'trial'])
        n_groups = len(groups)

        for g_idx, ((subj, sess, trial), trial_df) in enumerate(groups):
            if g_idx % 200 == 0:
                print(f"    Processing trial group {g_idx}/{n_groups} …")
            try:
                rows = process_trial_region(trial_df, region, sim_cache)
                all_rows.extend(rows)
            except Exception as e:
                print(f"    FAILED [{subj} sess={sess} trial={trial}]: {e}")
                traceback.print_exc()
                continue

        if not all_rows:
            print(f"  ✗ No pairs generated for region {region}")
            continue

        result_df = pd.DataFrame(all_rows, columns=OUTPUT_COLS)
        all_region_dfs.append(result_df)

        # Per-region × per-band outputs
        for band_name in BAND_ORDER:
            band_df  = result_df[result_df['band'] == band_name]
            out_path = OUTPUT_DIR / f"ALL_SUBJECTS_{exp}_{region}_{band_name}_rsa_lag_frac.csv"
            band_df.to_csv(out_path, index=False)
            print(f"    ✓ {region}/{band_name}: {len(band_df):,} pairs | "
                  f"RSA NaN={band_df['RSA_r'].isna().mean()*100:.1f}% | "
                  f"SemSim NaN={band_df['semantic_sim'].isna().mean()*100:.1f}% | "
                  f"→ {out_path.name}")

        # All-bands file for this region
        reg_path = OUTPUT_DIR / f"ALL_SUBJECTS_{exp}_{region}_allbands_rsa_lag_frac.csv"
        result_df.to_csv(reg_path, index=False)
        print(f"  ✓ {region} all-bands → {reg_path.name}")

    # Combined: both HC regions × all bands
    if all_region_dfs:
        combined  = pd.concat(all_region_dfs, ignore_index=True)
        comb_path = OUTPUT_DIR / f"ALL_SUBJECTS_{exp}_hc_allbands_rsa_lag_frac.csv"
        combined.to_csv(comb_path, index=False)
        print(f"\n  ✓ Combined HC all-bands → {comb_path.name}")
        print(f"    Total rows : {len(combined):,}")
        print(f"    Regions    : {combined['region'].unique().tolist()}")
        print(f"    Bands      : {combined['band'].unique().tolist()}")


# ============================================================================
# MAIN
# ============================================================================

if __name__ == '__main__':
    w2v_model = load_word2vec(WORD2VEC_PATH)

    for exp in EXPERIMENTS:
        run_experiment(exp, w2v_model)

    print(f"\n{'='*70}")
    print("✓ ALL EXPERIMENTS COMPLETE")
    print(f"{'='*70}")

  Loading Word2Vec from /home1/noaherz/word2vec/GoogleNews-vectors-negative300.bin.gz …
  ✓ Word2Vec loaded — vocab: 3,000,000

EXPERIMENT: DBOY1
  Loading subject_results_DBOY1/ALL_SUBJECTS_DBOY1_irasa_channel_wide.csv …
  Loaded 9,298 rows | 37 subjects | 6 sessions
  After HC filter: 6,173 rows
    Building semsim cache: 233 words → 27028 pairs …

  ────────────────────────────────────────────────────────────
  Region: LHP
  Rows: 3,609
    Processing trial group 0/179 …
    ✓ LHP/theta: 1,814 pairs | RSA NaN=0.0% | SemSim NaN=23.6% | → ALL_SUBJECTS_DBOY1_LHP_theta_rsa_lag_frac.csv
    ✓ LHP/alpha: 1,814 pairs | RSA NaN=0.0% | SemSim NaN=23.6% | → ALL_SUBJECTS_DBOY1_LHP_alpha_rsa_lag_frac.csv
    ✓ LHP/beta: 1,814 pairs | RSA NaN=0.0% | SemSim NaN=23.6% | → ALL_SUBJECTS_DBOY1_LHP_beta_rsa_lag_frac.csv
    ✓ LHP/gamma: 1,814 pairs | RSA NaN=0.0% | SemSim NaN=23.6% | → ALL_SUBJECTS_DBOY1_LHP_gamma_rsa_lag_frac.csv
  ✓ LHP all-bands → ALL_SUBJECTS_DBOY1_LHP_allbands_rsa_lag_frac.csv

 

In [2]:
#!/usr/bin/env python3
"""
RSA Lag — Sequential LMM Analysis (4 Steps) — RAW SCALE, DBOY1 only
========================================================================
*** FRACTAL (frac) variant — reads from rsa_lag_hc_bands_frac/ ***

Identical logic to the osc version but uses RSA_r values computed from
fractal (ret_frac_f*) IRASA vectors.

Betas are in original units:
  RSA_r      : Pearson r
  T_lag      : number of list positions separating the two items
  SP_lag     : Euclidean distance in store-coordinate units
  output_lag : number of recall output positions between the two items

Steps
-----
  Step 1 — RSA_r ~ predictor + output_lag + (1|subj)
            All bands pooled, both hemispheres.

  Step 2 — RSA_r ~ predictor * band (explicit dummies, theta = reference)
                 + output_lag + (1|subj)
            Tests whether predictor slope differs across bands.

  Step 3 — RSA_r ~ predictor * hemisphere + output_lag + (1|subj)
            Theta band only.  Tests LHP vs RHP asymmetry.
            Also run separately for LHP and RHP.

  Step 4 — RSA_r ~ T_lag + SP_lag + output_lag + (1|subj)
            Theta band only.  Tests independence of T_lag and SP_lag.

Input  : ./rsa_lag_hc_bands_frac/ALL_SUBJECTS_DBOY1_hc_allbands_rsa_lag_frac.csv
Output : ./rsa_lag_hc_bands_frac/SEQUENTIAL_LMM_RAW_DBOY1_frac_results.csv
         ./rsa_lag_hc_bands_frac/SEQUENTIAL_LMM_RAW_DBOY1_frac_results.txt
"""

import warnings
from pathlib import Path
from typing import List, Optional

import numpy as np
import pandas as pd
from statsmodels.regression.mixed_linear_model import MixedLM
from statsmodels.stats.multitest import fdrcorrection

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

INPUT_DIR  = Path('./rsa_lag_hc_bands_frac')   # ← frac folder
OUTPUT_DIR = Path('./rsa_lag_hc_bands_frac')   # ← frac folder

BANDS         = ['theta', 'alpha', 'beta', 'gamma']
NON_REF_BANDS = ['alpha', 'beta', 'gamma']   # theta = reference

OUTCOME = 'RSA_r'


# ============================================================================
# DATA LOADING
# ============================================================================

def load_data() -> Optional[pd.DataFrame]:
    fpath = INPUT_DIR / "ALL_SUBJECTS_DBOY1_hc_allbands_rsa_lag_frac.csv"   # ← frac
    if not fpath.exists():
        print(f"  ✗ File not found: {fpath}")
        return None

    df = pd.read_csv(fpath)
    print(f"  Loaded {fpath.name}  ({len(df):,} rows)")

    if 'hemisphere' not in df.columns:
        df['hemisphere'] = df['region'].map({'LHP': 'left', 'RHP': 'right'})

    df['hemisphere_dummy'] = (df['hemisphere'] == 'right').astype(int)

    print(f"\n  Total rows : {len(df):,}")
    print(f"  Subjects   : {df['subject'].nunique()}")
    print(f"  Sessions   : {df['session'].nunique()}")
    print(f"  Bands      : {sorted(df['band'].unique().tolist())}")
    print(f"  Regions    : {sorted(df['region'].unique().tolist())}")

    print(f"\n  Predictor scales (raw):")
    for col in ['RSA_r', 'T_lag', 'SP_lag', 'output_lag']:
        if col in df.columns:
            s = df[col].dropna()
            print(f"    {col:<12} mean={s.mean():.3f}  "
                  f"sd={s.std():.3f}  "
                  f"range=[{s.min():.2f}, {s.max():.2f}]")

    return df


# ============================================================================
# LMM FITTING
# ============================================================================

def fit_lmm(df: pd.DataFrame,
            formula_cols: List[str],
            formula_rhs: str,
            label: str) -> Optional[object]:
    """
    Fit RSA_r ~ formula_rhs.

    Random effects strategy (tried in order):
      1. Nested: (1|subject) + (1|subject:session)  via vc_formula
      2. Simple : (1|subject) only — fallback if nested hits a boundary
    """
    df = df.copy()
    df['subj_sess'] = df['subject'].astype(str) + '_' + df['session'].astype(str)

    keep = [OUTCOME] + [c for c in formula_cols if c in df.columns] \
           + ['subject', 'subj_sess']
    df   = df[keep].dropna()

    n = len(df)
    if n < 20:
        print(f"  [{label}] Too few rows ({n}) — skipping")
        return None

    formula = f"{OUTCOME} ~ {formula_rhs}"
    print(f"\n  [{label}] {formula}")
    print(f"  [{label}] N={n:,}  subjects={df['subject'].nunique()}")

    def _is_boundary(res) -> bool:
        if res is None or not np.isfinite(res.llf):
            return True
        for k in res.params.index:
            if 'subj_sess' in k.lower():
                if abs(res.params[k] - 16.0) < 1e-6:
                    return True
                if not np.isfinite(res.bse[k]):
                    return True
        return False

    # Attempt 1: nested random effects
    result = None
    for method in ['lbfgs', 'nm', 'powell']:
        try:
            model = MixedLM.from_formula(
                formula,
                data       = df,
                groups     = df['subject'],
                vc_formula = {'subj_sess': '0 + C(subj_sess)'},
            )
            res = model.fit(reml=True, method=method)
            if np.isfinite(res.llf) and not _is_boundary(res):
                print(f"  [{label}] nested RE  optimizer={method}  "
                      f"converged={getattr(res, 'converged', None)}")
                result = res
                break
        except Exception:
            pass

    # Attempt 2: subject-only random effects
    if result is None or _is_boundary(result):
        print(f"  [{label}] ⚠ Nested RE boundary — "
              f"falling back to (1|subject) only")
        for method in ['lbfgs', 'nm', 'powell']:
            try:
                model = MixedLM.from_formula(
                    formula,
                    data   = df,
                    groups = df['subject'],
                )
                res = model.fit(reml=True, method=method)
                if np.isfinite(res.llf):
                    print(f"  [{label}] subject-only RE  optimizer={method}  "
                          f"converged={getattr(res, 'converged', None)}")
                    result = res
                    break
            except Exception as e:
                print(f"  [{label}] subject-only {method} failed: {e}")
                result = None

    if result is None:
        print(f"  [{label}] WARNING: all fits unsuccessful")
    return result


# ============================================================================
# RESULT FORMATTING
# ============================================================================

def sig_stars(p: float) -> str:
    return ('***' if p < 0.001 else '**' if p < 0.01 else
            '*'   if p < 0.05  else '†'  if p < 0.10  else '')


def result_to_df(result, step: str, model_desc: str,
                 hemisphere: str = 'combined') -> pd.DataFrame:
    if result is None:
        return pd.DataFrame()

    rows = []
    for term in result.params.index:
        p_val = result.pvalues[term]
        is_re = (not np.isfinite(result.bse[term]) or
                 not np.isfinite(p_val) or
                 'var' in term.lower())
        rows.append({
            'step':       step,
            'model':      model_desc,
            'hemisphere': hemisphere,
            'term':       term,
            'beta':       result.params[term],
            'se':         result.bse[term],
            'z':          result.tvalues[term],
            'p_raw':      p_val,
            'p_fdr':      np.nan,
            'is_re':      is_re,
            'nobs':       int(result.nobs),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    fe_mask = (~df['is_re']) & df['p_raw'].notna() & np.isfinite(df['p_raw'])
    if fe_mask.sum() > 0:
        _, fdr_vals = fdrcorrection(df.loc[fe_mask, 'p_raw'].values)
        df.loc[fe_mask, 'p_fdr'] = fdr_vals

    return df


def print_table(df: pd.DataFrame, title: str):
    if df.empty:
        print(f"\n  [no results for: {title}]")
        return

    sep  = '=' * 95
    sep2 = '-' * 95
    hdr  = (f"{'Term':<45} {'β':>10} {'SE':>10} {'z':>8} "
            f"{'p_raw':>10} {'p_fdr':>10} {'N':>8}")

    print(f"\n{sep}")
    print(title)
    print(sep2)
    print(hdr)
    print(sep2)

    fe = df[~df['is_re']] if 'is_re' in df.columns else df
    for _, row in fe.iterrows():
        p_fdr = row.get('p_fdr', np.nan)
        p_fdr_str = f"{p_fdr:>10.4f}" if np.isfinite(p_fdr) else f"{'—':>10}"
        stars = sig_stars(p_fdr) if np.isfinite(p_fdr) else ''
        print(f"  {row['term']:<43} {row['beta']:>10.5f} {row['se']:>10.5f} "
              f"{row['z']:>8.3f} {row['p_raw']:>10.4f} "
              f"{p_fdr_str} {int(row['nobs']):>8,}  {stars}")

    re = df[df['is_re']] if 'is_re' in df.columns else pd.DataFrame()
    if not re.empty:
        print(f"  {'— random effects —'}")
        for _, row in re.iterrows():
            beta_str = f"{row['beta']:>10.5f}" if np.isfinite(row['beta']) \
                       else f"{'—':>10}"
            print(f"  {row['term']:<43} {beta_str}")

    print(sep2)
    print("  Outcome = RSA_r (raw Pearson r, fractal IRASA).  DBOY1 only.")
    print("  FDR: BH within fixed effects only.  "
          "† p<.10  * p<.05  ** p<.01  *** p<.001")
    print(f"  Reference: band=theta, hemisphere=LHP")
    print(sep)


# ============================================================================
# STEP 1
# ============================================================================

def step1(df: pd.DataFrame) -> List[pd.DataFrame]:
    print(f"\n{'#'*70}")
    print("STEP 1 — Does RSA_r track T_lag / SP_lag at all?")
    print(f"{'#'*70}")
    print("  Data: all bands pooled, both hemispheres")

    results = []
    for pred in ['T_lag', 'SP_lag']:
        res = fit_lmm(
            df,
            formula_cols = [pred, 'output_lag'],
            formula_rhs  = f'{pred} + output_lag',
            label        = f'Step1 {pred}',
        )
        rdf = result_to_df(res, step='Step1',
                           model_desc=f'RSA_r ~ {pred} + output_lag')
        print_table(rdf,
                    f"Step 1 | RSA_r ~ {pred} + output_lag  [all bands, combined]")
        results.append(rdf)

    return results


# ============================================================================
# STEP 2
# ============================================================================

def step2(df: pd.DataFrame) -> List[pd.DataFrame]:
    print(f"\n{'#'*70}")
    print("STEP 2 — Is the effect theta-specific?")
    print(f"{'#'*70}")
    print("  Data: all bands pooled, both hemispheres")
    print("  Reference band: theta")

    df = df.copy()
    for band in NON_REF_BANDS:
        df[f'band_{band}'] = (df['band'] == band).astype(float)

    results = []
    for pred in ['T_lag', 'SP_lag']:

        for band in NON_REF_BANDS:
            df[f'{pred}_x_{band}'] = df[pred] * df[f'band_{band}']

        band_cols  = [f'band_{b}'     for b in NON_REF_BANDS]
        inter_cols = [f'{pred}_x_{b}' for b in NON_REF_BANDS]

        formula_cols = [pred] + band_cols + inter_cols + ['output_lag']
        formula_rhs  = (f'{pred} + '
                        + ' + '.join(band_cols) + ' + '
                        + ' + '.join(inter_cols)
                        + ' + output_lag')

        res = fit_lmm(df, formula_cols, formula_rhs, label=f'Step2 {pred}')
        rdf = result_to_df(res, step='Step2',
                           model_desc=f'RSA_r ~ {pred} * band + output_lag')
        print_table(rdf,
                    f"Step 2 | RSA_r ~ {pred} * band + output_lag  "
                    f"[all bands, combined]\n"
                    f"  KEY: {pred}_x_{{band}} significant → slope ≠ theta")
        results.append(rdf)

    return results


# ============================================================================
# STEP 3
# ============================================================================

def step3(df: pd.DataFrame) -> List[pd.DataFrame]:
    print(f"\n{'#'*70}")
    print("STEP 3 — Is the theta effect lateralized?  (theta band only)")
    print(f"{'#'*70}")

    theta_df = df[df['band'] == 'theta'].copy()
    print(f"  Theta rows: {len(theta_df):,}  "
          f"subjects: {theta_df['subject'].nunique()}")

    results = []
    for pred in ['T_lag', 'SP_lag']:
        inter_col = f'{pred}_x_hemi'
        theta_df[inter_col] = theta_df[pred] * theta_df['hemisphere_dummy']

        # Combined: interaction test
        res = fit_lmm(
            theta_df,
            formula_cols = [pred, 'hemisphere_dummy', inter_col, 'output_lag'],
            formula_rhs  = (f'{pred} + hemisphere_dummy + {inter_col}'
                            f' + output_lag'),
            label        = f'Step3 {pred} combined',
        )
        rdf = result_to_df(res, step='Step3',
                           model_desc=f'RSA_r ~ {pred} * hemisphere + output_lag',
                           hemisphere='combined')
        print_table(rdf,
                    f"Step 3 | RSA_r ~ {pred} * hemisphere + output_lag  "
                    f"[theta, combined]\n"
                    f"  KEY TERM: {inter_col}  "
                    f"(significant → LHP slope ≠ RHP slope)")
        results.append(rdf)

        # Simple slopes per hemisphere
        for hemi, hemi_label in [('left', 'LHP'), ('right', 'RHP')]:
            hd = theta_df[theta_df['hemisphere'] == hemi].copy()
            res_h = fit_lmm(
                hd,
                formula_cols = [pred, 'output_lag'],
                formula_rhs  = f'{pred} + output_lag',
                label        = f'Step3 {pred} {hemi_label}',
            )
            rdf_h = result_to_df(res_h, step='Step3',
                                 model_desc=f'RSA_r ~ {pred} + output_lag',
                                 hemisphere=hemi_label)
            print_table(rdf_h,
                        f"Step 3 | RSA_r ~ {pred} + output_lag  "
                        f"[theta, {hemi_label} only]")
            results.append(rdf_h)

    return results


# ============================================================================
# STEP 4
# ============================================================================

def step4(df: pd.DataFrame) -> List[pd.DataFrame]:
    print(f"\n{'#'*70}")
    print("STEP 4 — Do T_lag and SP_lag survive mutual control?  (theta only)")
    print(f"{'#'*70}")

    theta_df = df[df['band'] == 'theta'].copy()
    print(f"  Theta rows: {len(theta_df):,}  "
          f"subjects: {theta_df['subject'].nunique()}")

    formula_cols = ['T_lag', 'SP_lag', 'output_lag']
    formula_rhs  = 'T_lag + SP_lag + output_lag'

    results = []
    for subset_label, subset_df in [
            ('combined', theta_df),
            ('LHP',      theta_df[theta_df['hemisphere'] == 'left']),
            ('RHP',      theta_df[theta_df['hemisphere'] == 'right'])]:

        res = fit_lmm(
            subset_df,
            formula_cols = formula_cols,
            formula_rhs  = formula_rhs,
            label        = f'Step4 {subset_label}',
        )
        rdf = result_to_df(res, step='Step4',
                           model_desc='RSA_r ~ T_lag + SP_lag + output_lag',
                           hemisphere=subset_label)
        print_table(rdf,
                    f"Step 4 | RSA_r ~ T_lag + SP_lag + output_lag  "
                    f"[theta, {subset_label}]")
        results.append(rdf)

    return results


# ============================================================================
# ENTRY POINT
# ============================================================================

def main():
    print(f"\n{'='*70}")
    print("RSA LAG — SEQUENTIAL LMM  |  DBOY1 only  |  RAW SCALE  |  FRAC")
    print(f"{'='*70}")

    df = load_data()
    if df is None or df.empty:
        print("No data loaded. Exiting.")
        return

    all_results = []
    all_results += step1(df)
    all_results += step2(df)
    all_results += step3(df)
    all_results += step4(df)

    # Save CSV
    if all_results:
        master = pd.concat(all_results, ignore_index=True)
        csv_path = OUTPUT_DIR / 'SEQUENTIAL_LMM_RAW_DBOY1_frac_results.csv'   # ← frac
        master.to_csv(csv_path, index=False)
        print(f"\n  ✓ Results → {csv_path}  ({len(master):,} rows)")

        # Save text
        txt_path = OUTPUT_DIR / 'SEQUENTIAL_LMM_RAW_DBOY1_frac_results.txt'   # ← frac
        lines = []
        for rdf in all_results:
            if rdf.empty:
                continue
            lines.append(f"\n{'='*80}")
            lines.append(f"{rdf['step'].iloc[0]}  |  "
                         f"{rdf['model'].iloc[0]}  |  "
                         f"{rdf['hemisphere'].iloc[0]}")
            lines.append(f"{'='*80}")
            for _, row in rdf[~rdf['is_re']].iterrows():
                p_fdr = row['p_fdr']
                stars = sig_stars(p_fdr) if np.isfinite(p_fdr) else ''
                p_fdr_s = f"{p_fdr:.4f}" if np.isfinite(p_fdr) else '—'
                lines.append(
                    f"  {row['term']:<40} β={row['beta']:>10.5f}  "
                    f"SE={row['se']:>9.5f}  z={row['z']:>7.3f}  "
                    f"p={row['p_raw']:>8.4f}  p_fdr={p_fdr_s:>8}  {stars}"
                )
        with open(txt_path, 'w') as f:
            f.write('\n'.join(lines))
        print(f"  ✓ Text    → {txt_path}")

    print(f"\n{'='*70}")
    print("INTERPRETATION GUIDE")
    print(f"{'='*70}")
    print("""
  Step 1: T_lag/SP_lag significant controlling for output_lag?
          → genuine clustering effect, not recall-order artifact.

  Step 2: predictor_x_alpha/beta/gamma significant?
          → theta slope differs from that band → theta specificity confirmed.
          NOT significant → effect is broadband, not theta-specific.

  Step 3: predictor_x_hemi significant?
          → LHP and RHP slopes are genuinely different.
          Simple slopes show which hemisphere drives the effect.

  Step 4: Both T_lag and SP_lag survive together + output_lag?
          → temporal and spatial clustering are INDEPENDENT.
    """)


if __name__ == '__main__':
    main()


RSA LAG — SEQUENTIAL LMM  |  DBOY1 only  |  RAW SCALE  |  FRAC
  Loaded ALL_SUBJECTS_DBOY1_hc_allbands_rsa_lag_frac.csv  (13,028 rows)

  Total rows : 13,028
  Subjects   : 34
  Sessions   : 6
  Bands      : ['alpha', 'beta', 'gamma', 'theta']
  Regions    : ['LHP', 'RHP']

  Predictor scales (raw):
    RSA_r        mean=0.820  sd=0.254  range=[-1.00, 1.00]
    T_lag        mean=4.523  sd=2.914  range=[1.00, 11.00]
    SP_lag       mean=70.149  sd=30.696  range=[12.96, 142.72]
    output_lag   mean=2.450  sd=1.560  range=[1.00, 10.00]

######################################################################
STEP 1 — Does RSA_r track T_lag / SP_lag at all?
######################################################################
  Data: all bands pooled, both hemispheres

  [Step1 T_lag] RSA_r ~ T_lag + output_lag
  [Step1 T_lag] N=13,028  subjects=34
  [Step1 T_lag] nested RE  optimizer=nm  converged=True

Step 1 | RSA_r ~ T_lag + output_lag  [all bands, combined]
-------------------------

In [3]:
#!/usr/bin/env python3
"""
RSA Lag Analysis — Hippocampus Only (LHP / RHP), Band × Pair (Long Format)
---------------------------------------------------------------------------
*** MIXED POWER variant (frac + osc → mixed, then band-split RSA) ***

Pipeline per recall event:
  Step 1 — Reconstruct mixed power per channel per frequency:
               mixed[ch, freq] = ret_frac_f{freq} + ret_osc_f{freq}

  Step 2 — Slice to band frequency indices → (N_band_freqs × N_channels) matrix
            (freq is the OUTER / row dimension, channel is the INNER / col dimension)

  Step 3 — Flatten row-major → 1-D vector of length (N_band_freqs × N_channels)

  Step 4 — RSA_r = Pearson r between vectors of recall i and recall j
            (NaN elements masked pairwise; require ≥ 3 finite pairs)

Regions : LHP, RHP
Bands   : theta  (freq idx 0–3,   ~3–6 Hz)
          alpha  (freq idx 4–6,   ~7–11 Hz)
          beta   (freq idx 7–11,  ~14–34 Hz)
          gamma  (freq idx 12–17, ~42–128 Hz)

18 log-spaced frequencies (3–128 Hz):
  idx:  0     1     2     3     4     5      6      7      8
  Hz:  3.00  3.74  4.67  5.82  7.26  9.05  11.28  14.07  17.55
  idx:  9     10    11    12    13    14    15     16     17
  Hz: 21.88 27.29 34.03 42.44 52.92 66.00 82.31 102.64 128.00

Output — LONG format: one row per pair × band.

Columns:
  subject, session, experiment, trial,
  region, hemisphere,
  band, band_freq_indices,
  output_pos_i, output_pos_j, output_lag,
  word_i, word_j,
  serial_pos_i, serial_pos_j, T_lag, SP_lag,
  RSA_r, n_channels, semantic_sim

Output files:
  ./rsa_lag_hc_bands_mixed/ALL_SUBJECTS_{exp}_{region}_{band}_rsa_lag_mixed.csv
  ./rsa_lag_hc_bands_mixed/ALL_SUBJECTS_{exp}_{region}_allbands_rsa_lag_mixed.csv
  ./rsa_lag_hc_bands_mixed/ALL_SUBJECTS_{exp}_hc_allbands_rsa_lag_mixed.csv
"""

import warnings
import traceback
from pathlib import Path
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from scipy.spatial.distance import euclidean

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

EXPERIMENTS = ['DBOY1', 'EFRCourierReadOnly', 'EFRCourierOpenLoop']

INPUT_DIRS = {
    'DBOY1':               Path('./subject_results_DBOY1'),
    'EFRCourierReadOnly':  Path('./subject_results_EFRCourierReadOnly'),
    'EFRCourierOpenLoop':  Path('./subject_results_EFRCourierOpenLoop'),
}

OUTPUT_DIR = Path('./rsa_lag_hc_bands_mixed')
OUTPUT_DIR.mkdir(exist_ok=True)

# Hippocampus only
REGIONS    = ['LHP', 'RHP']
HEMISPHERE = {'LHP': 'left', 'RHP': 'right'}

# 18 log-spaced frequencies 3–128 Hz
N_FREQS = 18

BANDS = {
    'theta': list(range(0, 4)),    # 3.00 – 5.82 Hz
    'alpha': list(range(4, 7)),    # 7.26 – 11.28 Hz
    'beta':  list(range(7, 12)),   # 14.07 – 34.03 Hz
    'gamma': list(range(12, 18)),  # 42.44 – 128.00 Hz
}
BAND_ORDER = ['theta', 'alpha', 'beta', 'gamma']

# Source columns (fractal and oscillatory SSL power)
RET_FRAC_COLS = [f'ret_frac_f{i:02d}' for i in range(N_FREQS)]
RET_OSC_COLS  = [f'ret_osc_f{i:02d}'  for i in range(N_FREQS)]

# ---- Word2Vec ---------------------------------------------------------------
WORD2VEC_PATH = Path('/home1/noaherz/word2vec/GoogleNews-vectors-negative300.bin.gz')

OUTPUT_COLS = [
    'subject', 'session', 'experiment', 'trial',
    'region', 'hemisphere',
    'band', 'band_freq_indices',
    'output_pos_i', 'output_pos_j', 'output_lag',
    'word_i', 'word_j',
    'serial_pos_i', 'serial_pos_j', 'T_lag', 'SP_lag',
    'RSA_r', 'n_channels', 'semantic_sim',
]

# ============================================================================
# WORD2VEC LOADER + SIMILARITY
# ============================================================================

def load_word2vec(path: Path):
    if path is None or not path.exists():
        print(f"  ⚠  Word2Vec model not found at {path}. semantic_sim will be NaN.")
        return None
    try:
        import gensim.models as gensim_models
        print(f"  Loading Word2Vec from {path} …")
        model = gensim_models.KeyedVectors.load_word2vec_format(str(path), binary=True)
        try:
            vs = len(model)
        except TypeError:
            vs = len(model.vocab)
        print(f"  ✓ Word2Vec loaded — vocab: {vs:,}")
        return model
    except Exception as e:
        print(f"  ✗ Word2Vec load failed: {e}. semantic_sim will be NaN.")
        return None


def case_insensitive_similarity(word1: str, word2: str, model) -> Optional[float]:
    cases = [
        (word1.lower(), word2.lower()),
        (word1.lower(), word2.upper()),
        (word1.upper(), word2.lower()),
        (word1.upper(), word2.upper()),
    ]
    sims = []
    for w1, w2 in cases:
        try:
            sims.append(model.similarity(w1, w2))
        except KeyError:
            continue
    return float(max(sims)) if sims else None


def build_similarity_cache(words: set, model) -> dict:
    if model is None:
        return {}
    unique_words = sorted(w for w in words if isinstance(w, str))
    n = len(unique_words)
    print(f"    Building semsim cache: {n} words → {n*(n-1)//2} pairs …")
    cache = {}
    for i, w1 in enumerate(unique_words):
        for w2 in unique_words[i:]:
            key = frozenset({w1, w2})
            if key not in cache:
                sim = case_insensitive_similarity(w1, w2, model)
                cache[key] = sim if sim is not None else np.nan
    return cache


# ============================================================================
# VECTOR + STATISTICS HELPERS
# ============================================================================

def build_mixed_band_vector(recall_rows: pd.DataFrame,
                            channel_index,
                            band_freq_indices: list) -> np.ndarray:
    """
    Build the representational vector for one recall event and one band.

    Steps:
      1. Align rows to canonical channel_index → (N_ch,) indexed DataFrame
      2. Compute mixed = frac + osc for all 18 frequencies
         → mixed_mat shape: (N_ch, N_freqs)
      3. Slice to band_freq_indices
         → band_mat shape: (N_ch, N_band_freqs)
      4. Transpose to (N_band_freqs, N_ch)   ← freq is outer dimension
      5. Flatten row-major → 1-D vector of length N_band_freqs * N_ch

    Missing channels (not in recall_rows) retain NaN — handled in safe_pearsonr.
    """
    # Step 1: align to canonical channel order
    ch_df = (
        recall_rows
        .drop_duplicates(subset='channel_idx')
        .set_index('channel_idx')
        .reindex(channel_index)
    )

    # Step 2: mixed power = frac + osc, shape (N_ch, N_freqs)
    frac_mat = ch_df[RET_FRAC_COLS].values   # (N_ch, 18)
    osc_mat  = ch_df[RET_OSC_COLS].values    # (N_ch, 18)
    mixed_mat = frac_mat + osc_mat           # (N_ch, 18)  — element-wise sum

    # Step 3: slice to band frequencies → (N_ch, N_band_freqs)
    band_mat = mixed_mat[:, band_freq_indices]

    # Step 4: transpose → (N_band_freqs, N_ch)
    band_mat_T = band_mat.T

    # Step 5: flatten row-major → (N_band_freqs * N_ch,)
    return band_mat_T.flatten()


def safe_pearsonr(v1: np.ndarray, v2: np.ndarray) -> float:
    if len(v1) != len(v2):
        return np.nan
    mask = np.isfinite(v1) & np.isfinite(v2)
    if mask.sum() < 3:
        return np.nan
    r, _ = pearsonr(v1[mask], v2[mask])
    return float(r)


def safe_euclidean(x1, z1, x2, z2) -> float:
    if any(not np.isfinite(v) for v in (x1, z1, x2, z2)):
        return np.nan
    return float(euclidean([x1, z1], [x2, z2]))


def extract_scalar(series: pd.Series, field: str, context: str):
    unique_vals = series.dropna().unique()
    if len(unique_vals) > 1:
        warnings.warn(
            f"[{context}] '{field}' has {len(unique_vals)} distinct values "
            f"({unique_vals[:3]}…). Taking first.")
    return series.iloc[0]


# ============================================================================
# TRIAL-LEVEL PROCESSING
# ============================================================================

def process_trial_region(trial_df: pd.DataFrame,
                         region: str,
                         sim_cache: dict) -> List[Dict]:
    """
    For one (subject, session, trial, region):
      - Enumerate all valid recall pairs (i, j), i < j
      - For each pair × each band: compute RSA_r on the mixed-power band vector
      - Returns list of dicts (long format, one row per pair × band)
    """
    rows = []

    output_positions = sorted(
        trial_df['recall_output_position'].unique(),
        key=lambda x: int(x),
    )
    if len(output_positions) < 2:
        return rows

    channel_index = sorted(trial_df['channel_idx'].unique(), key=int)
    sample_row    = trial_df.iloc[0]

    # ------------------------------------------------------------------
    # Pre-compute per-output-position metadata + per-band mixed vectors
    # ------------------------------------------------------------------
    pos_data: Dict[int, Dict] = {}

    for op in output_positions:
        op_rows = trial_df[trial_df['recall_output_position'] == op]
        ctx = (f"subj={sample_row['subject']} sess={sample_row['session']} "
               f"trial={sample_row['trial']} region={region} op={op}")

        word       = extract_scalar(op_rows['recalled_word'],   'recalled_word',   ctx)
        serial_pos = extract_scalar(op_rows['serial_position'], 'serial_position', ctx)
        store_x    = extract_scalar(op_rows['store_x'],         'store_x',         ctx)
        store_z    = extract_scalar(op_rows['store_z'],         'store_z',         ctx)
        n_ch       = op_rows['channel_idx'].nunique()

        # Build one mixed-power vector per band
        band_vectors = {
            band_name: build_mixed_band_vector(op_rows, channel_index, freq_idx)
            for band_name, freq_idx in BANDS.items()
        }

        pos_data[op] = {
            'band_vectors': band_vectors,
            'word':         word,
            'serial_pos':   serial_pos,
            'store_x':      store_x,
            'store_z':      store_z,
            'n_channels':   n_ch,
        }

    subject    = sample_row['subject']
    session    = sample_row['session']
    experiment = sample_row['experiment']
    trial      = sample_row['trial']

    # ------------------------------------------------------------------
    # Enumerate pairs × bands
    # ------------------------------------------------------------------
    for idx_i, op_i in enumerate(output_positions):
        for op_j in output_positions[idx_i + 1:]:
            d_i = pos_data[op_i]
            d_j = pos_data[op_j]

            output_lag = int(op_j) - int(op_i)
            T_lag      = abs(int(d_i['serial_pos']) - int(d_j['serial_pos']))
            SP_lag     = safe_euclidean(d_i['store_x'], d_i['store_z'],
                                        d_j['store_x'], d_j['store_z'])
            n_channels = min(d_i['n_channels'], d_j['n_channels'])

            # Semantic similarity — same for all bands, compute once per pair
            w_i = str(d_i['word']).lower() if pd.notna(d_i['word']) else None
            w_j = str(d_j['word']).lower() if pd.notna(d_j['word']) else None
            sem_sim = (
                sim_cache.get(frozenset({w_i, w_j}), np.nan)
                if (w_i and w_j and sim_cache)
                else np.nan
            )

            # One row per band
            for band_name in BAND_ORDER:
                freq_idx = BANDS[band_name]
                v_i      = d_i['band_vectors'][band_name]
                v_j      = d_j['band_vectors'][band_name]
                rsa_r    = safe_pearsonr(v_i, v_j)

                rows.append({
                    'subject':           subject,
                    'session':           session,
                    'experiment':        experiment,
                    'trial':             trial,
                    'region':            region,
                    'hemisphere':        HEMISPHERE[region],
                    'band':              band_name,
                    'band_freq_indices': str(freq_idx),
                    'output_pos_i':      op_i,
                    'output_pos_j':      op_j,
                    'output_lag':        output_lag,
                    'word_i':            d_i['word'],
                    'word_j':            d_j['word'],
                    'serial_pos_i':      d_i['serial_pos'],
                    'serial_pos_j':      d_j['serial_pos'],
                    'T_lag':             T_lag,
                    'SP_lag':            SP_lag,
                    'RSA_r':             rsa_r,
                    'n_channels':        n_channels,
                    'semantic_sim':      sem_sim,
                })

    return rows


# ============================================================================
# PER-EXPERIMENT RUNNER
# ============================================================================

def run_experiment(exp: str, w2v_model):
    print(f"\n{'='*70}")
    print(f"EXPERIMENT: {exp}")
    print(f"{'='*70}")

    input_dir  = INPUT_DIRS.get(exp)
    if input_dir is None:
        print(f"  ✗ No INPUT_DIR configured for '{exp}'.")
        return

    input_path = input_dir / f"ALL_SUBJECTS_{exp}_irasa_channel_wide.csv"
    if not input_path.exists():
        print(f"  ✗ Master CSV not found: {input_path}")
        return

    print(f"  Loading {input_path} …")
    df = pd.read_csv(input_path)
    print(f"  Loaded {len(df):,} rows | "
          f"{df['subject'].nunique()} subjects | "
          f"{df['session'].nunique()} sessions")

    # Verify required columns exist
    missing_frac = [c for c in RET_FRAC_COLS if c not in df.columns]
    missing_osc  = [c for c in RET_OSC_COLS  if c not in df.columns]
    if missing_frac or missing_osc:
        print(f"  ✗ Missing columns — frac: {missing_frac[:3]}{'…' if len(missing_frac)>3 else ''} "
              f"| osc: {missing_osc[:3]}{'…' if len(missing_osc)>3 else ''}")
        return

    # Filter to HC only
    df = df[df['region'].isin(REGIONS)].copy()
    print(f"  After HC filter: {len(df):,} rows")
    if df.empty:
        print("  ✗ No hippocampal data found — check 'region' column values.")
        return

    df['channel_idx'] = df['channel_idx'].astype(int)

    # Build Word2Vec cache once per experiment
    all_words_lower = set(
        df['recalled_word'].dropna().astype(str).str.lower().unique()
    )
    sim_cache = build_similarity_cache(all_words_lower, w2v_model)

    all_region_dfs = []

    for region in REGIONS:
        print(f"\n  {'─'*60}")
        print(f"  Region: {region}")

        region_df = df[df['region'] == region].copy()
        if region_df.empty:
            print(f"  ✗ No rows for region {region} — skipping")
            continue

        print(f"  Rows: {len(region_df):,}")

        all_rows = []
        groups   = region_df.groupby(['subject', 'session', 'trial'])
        n_groups = len(groups)

        for g_idx, ((subj, sess, trial), trial_df) in enumerate(groups):
            if g_idx % 200 == 0:
                print(f"    Processing trial group {g_idx}/{n_groups} …")
            try:
                rows = process_trial_region(trial_df, region, sim_cache)
                all_rows.extend(rows)
            except Exception as e:
                print(f"    FAILED [{subj} sess={sess} trial={trial}]: {e}")
                traceback.print_exc()
                continue

        if not all_rows:
            print(f"  ✗ No pairs generated for region {region}")
            continue

        result_df = pd.DataFrame(all_rows, columns=OUTPUT_COLS)
        all_region_dfs.append(result_df)

        # Per-region × per-band outputs
        for band_name in BAND_ORDER:
            band_df  = result_df[result_df['band'] == band_name]
            out_path = OUTPUT_DIR / f"ALL_SUBJECTS_{exp}_{region}_{band_name}_rsa_lag_mixed.csv"
            band_df.to_csv(out_path, index=False)
            print(f"    ✓ {region}/{band_name}: {len(band_df):,} pairs | "
                  f"RSA NaN={band_df['RSA_r'].isna().mean()*100:.1f}% | "
                  f"SemSim NaN={band_df['semantic_sim'].isna().mean()*100:.1f}% | "
                  f"→ {out_path.name}")

        # All-bands file for this region
        reg_path = OUTPUT_DIR / f"ALL_SUBJECTS_{exp}_{region}_allbands_rsa_lag_mixed.csv"
        result_df.to_csv(reg_path, index=False)
        print(f"  ✓ {region} all-bands → {reg_path.name}")

    # Combined: both HC regions × all bands
    if all_region_dfs:
        combined  = pd.concat(all_region_dfs, ignore_index=True)
        comb_path = OUTPUT_DIR / f"ALL_SUBJECTS_{exp}_hc_allbands_rsa_lag_mixed.csv"
        combined.to_csv(comb_path, index=False)
        print(f"\n  ✓ Combined HC all-bands → {comb_path.name}")
        print(f"    Total rows : {len(combined):,}")
        print(f"    Regions    : {combined['region'].unique().tolist()}")
        print(f"    Bands      : {combined['band'].unique().tolist()}")


# ============================================================================
# MAIN
# ============================================================================

if __name__ == '__main__':
    w2v_model = load_word2vec(WORD2VEC_PATH)

    for exp in EXPERIMENTS:
        run_experiment(exp, w2v_model)

    print(f"\n{'='*70}")
    print("✓ ALL EXPERIMENTS COMPLETE")
    print(f"{'='*70}")

  Loading Word2Vec from /home1/noaherz/word2vec/GoogleNews-vectors-negative300.bin.gz …
  ✓ Word2Vec loaded — vocab: 3,000,000

EXPERIMENT: DBOY1
  Loading subject_results_DBOY1/ALL_SUBJECTS_DBOY1_irasa_channel_wide.csv …
  Loaded 9,298 rows | 37 subjects | 6 sessions
  After HC filter: 6,173 rows
    Building semsim cache: 233 words → 27028 pairs …

  ────────────────────────────────────────────────────────────
  Region: LHP
  Rows: 3,609
    Processing trial group 0/179 …
    ✓ LHP/theta: 1,814 pairs | RSA NaN=0.0% | SemSim NaN=23.6% | → ALL_SUBJECTS_DBOY1_LHP_theta_rsa_lag_mixed.csv
    ✓ LHP/alpha: 1,814 pairs | RSA NaN=0.0% | SemSim NaN=23.6% | → ALL_SUBJECTS_DBOY1_LHP_alpha_rsa_lag_mixed.csv
    ✓ LHP/beta: 1,814 pairs | RSA NaN=0.0% | SemSim NaN=23.6% | → ALL_SUBJECTS_DBOY1_LHP_beta_rsa_lag_mixed.csv
    ✓ LHP/gamma: 1,814 pairs | RSA NaN=0.0% | SemSim NaN=23.6% | → ALL_SUBJECTS_DBOY1_LHP_gamma_rsa_lag_mixed.csv
  ✓ LHP all-bands → ALL_SUBJECTS_DBOY1_LHP_allbands_rsa_lag_mixed.c

In [4]:
#!/usr/bin/env python3
"""
RSA Lag — Sequential LMM Analysis (4 Steps) — RAW SCALE, DBOY1 only
========================================================================
*** MIXED POWER variant (frac + osc) — reads from rsa_lag_hc_bands_mixed/ ***

Representational vectors were built as:
    mixed[freq, ch] = ret_frac_f{freq} + ret_osc_f{freq}
then sliced to band frequency indices (freq × channel, flattened).

Betas are in original units:
  RSA_r      : Pearson r
  T_lag      : number of list positions separating the two items
  SP_lag     : Euclidean distance in store-coordinate units
  output_lag : number of recall output positions between the two items

Steps
-----
  Step 1 — RSA_r ~ predictor + output_lag + (1|subj)
            All bands pooled, both hemispheres.

  Step 2 — RSA_r ~ predictor * band (explicit dummies, theta = reference)
                 + output_lag + (1|subj)
            Tests whether predictor slope differs across bands.

  Step 3 — RSA_r ~ predictor * hemisphere + output_lag + (1|subj)
            Theta band only.  Tests LHP vs RHP asymmetry.
            Also run separately for LHP and RHP.

  Step 4 — RSA_r ~ T_lag + SP_lag + output_lag + (1|subj)
            Theta band only.  Tests independence of T_lag and SP_lag.

Input  : ./rsa_lag_hc_bands_mixed/ALL_SUBJECTS_DBOY1_hc_allbands_rsa_lag_mixed.csv
Output : ./rsa_lag_hc_bands_mixed/SEQUENTIAL_LMM_RAW_DBOY1_mixed_results.csv
         ./rsa_lag_hc_bands_mixed/SEQUENTIAL_LMM_RAW_DBOY1_mixed_results.txt
"""

import warnings
from pathlib import Path
from typing import List, Optional

import numpy as np
import pandas as pd
from statsmodels.regression.mixed_linear_model import MixedLM
from statsmodels.stats.multitest import fdrcorrection

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

INPUT_DIR  = Path('./rsa_lag_hc_bands_mixed')   # ← mixed folder
OUTPUT_DIR = Path('./rsa_lag_hc_bands_mixed')   # ← mixed folder

BANDS         = ['theta', 'alpha', 'beta', 'gamma']
NON_REF_BANDS = ['alpha', 'beta', 'gamma']   # theta = reference

OUTCOME = 'RSA_r'


# ============================================================================
# DATA LOADING
# ============================================================================

def load_data() -> Optional[pd.DataFrame]:
    fpath = INPUT_DIR / "ALL_SUBJECTS_DBOY1_hc_allbands_rsa_lag_mixed.csv"   # ← mixed
    if not fpath.exists():
        print(f"  ✗ File not found: {fpath}")
        return None

    df = pd.read_csv(fpath)
    print(f"  Loaded {fpath.name}  ({len(df):,} rows)")

    if 'hemisphere' not in df.columns:
        df['hemisphere'] = df['region'].map({'LHP': 'left', 'RHP': 'right'})

    df['hemisphere_dummy'] = (df['hemisphere'] == 'right').astype(int)

    print(f"\n  Total rows : {len(df):,}")
    print(f"  Subjects   : {df['subject'].nunique()}")
    print(f"  Sessions   : {df['session'].nunique()}")
    print(f"  Bands      : {sorted(df['band'].unique().tolist())}")
    print(f"  Regions    : {sorted(df['region'].unique().tolist())}")

    print(f"\n  Predictor scales (raw):")
    for col in ['RSA_r', 'T_lag', 'SP_lag', 'output_lag']:
        if col in df.columns:
            s = df[col].dropna()
            print(f"    {col:<12} mean={s.mean():.3f}  "
                  f"sd={s.std():.3f}  "
                  f"range=[{s.min():.2f}, {s.max():.2f}]")

    return df


# ============================================================================
# LMM FITTING
# ============================================================================

def fit_lmm(df: pd.DataFrame,
            formula_cols: List[str],
            formula_rhs: str,
            label: str) -> Optional[object]:
    """
    Fit RSA_r ~ formula_rhs.

    Random effects strategy (tried in order):
      1. Nested: (1|subject) + (1|subject:session)  via vc_formula
      2. Simple : (1|subject) only — fallback if nested hits a boundary
    """
    df = df.copy()
    df['subj_sess'] = df['subject'].astype(str) + '_' + df['session'].astype(str)

    keep = [OUTCOME] + [c for c in formula_cols if c in df.columns] \
           + ['subject', 'subj_sess']
    df   = df[keep].dropna()

    n = len(df)
    if n < 20:
        print(f"  [{label}] Too few rows ({n}) — skipping")
        return None

    formula = f"{OUTCOME} ~ {formula_rhs}"
    print(f"\n  [{label}] {formula}")
    print(f"  [{label}] N={n:,}  subjects={df['subject'].nunique()}")

    def _is_boundary(res) -> bool:
        if res is None or not np.isfinite(res.llf):
            return True
        for k in res.params.index:
            if 'subj_sess' in k.lower():
                if abs(res.params[k] - 16.0) < 1e-6:
                    return True
                if not np.isfinite(res.bse[k]):
                    return True
        return False

    # Attempt 1: nested random effects
    result = None
    for method in ['lbfgs', 'nm', 'powell']:
        try:
            model = MixedLM.from_formula(
                formula,
                data       = df,
                groups     = df['subject'],
                vc_formula = {'subj_sess': '0 + C(subj_sess)'},
            )
            res = model.fit(reml=True, method=method)
            if np.isfinite(res.llf) and not _is_boundary(res):
                print(f"  [{label}] nested RE  optimizer={method}  "
                      f"converged={getattr(res, 'converged', None)}")
                result = res
                break
        except Exception:
            pass

    # Attempt 2: subject-only random effects
    if result is None or _is_boundary(result):
        print(f"  [{label}] ⚠ Nested RE boundary — "
              f"falling back to (1|subject) only")
        for method in ['lbfgs', 'nm', 'powell']:
            try:
                model = MixedLM.from_formula(
                    formula,
                    data   = df,
                    groups = df['subject'],
                )
                res = model.fit(reml=True, method=method)
                if np.isfinite(res.llf):
                    print(f"  [{label}] subject-only RE  optimizer={method}  "
                          f"converged={getattr(res, 'converged', None)}")
                    result = res
                    break
            except Exception as e:
                print(f"  [{label}] subject-only {method} failed: {e}")
                result = None

    if result is None:
        print(f"  [{label}] WARNING: all fits unsuccessful")
    return result


# ============================================================================
# RESULT FORMATTING
# ============================================================================

def sig_stars(p: float) -> str:
    return ('***' if p < 0.001 else '**' if p < 0.01 else
            '*'   if p < 0.05  else '†'  if p < 0.10  else '')


def result_to_df(result, step: str, model_desc: str,
                 hemisphere: str = 'combined') -> pd.DataFrame:
    if result is None:
        return pd.DataFrame()

    rows = []
    for term in result.params.index:
        p_val = result.pvalues[term]
        is_re = (not np.isfinite(result.bse[term]) or
                 not np.isfinite(p_val) or
                 'var' in term.lower())
        rows.append({
            'step':       step,
            'model':      model_desc,
            'hemisphere': hemisphere,
            'term':       term,
            'beta':       result.params[term],
            'se':         result.bse[term],
            'z':          result.tvalues[term],
            'p_raw':      p_val,
            'p_fdr':      np.nan,
            'is_re':      is_re,
            'nobs':       int(result.nobs),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    fe_mask = (~df['is_re']) & df['p_raw'].notna() & np.isfinite(df['p_raw'])
    if fe_mask.sum() > 0:
        _, fdr_vals = fdrcorrection(df.loc[fe_mask, 'p_raw'].values)
        df.loc[fe_mask, 'p_fdr'] = fdr_vals

    return df


def print_table(df: pd.DataFrame, title: str):
    if df.empty:
        print(f"\n  [no results for: {title}]")
        return

    sep  = '=' * 95
    sep2 = '-' * 95
    hdr  = (f"{'Term':<45} {'β':>10} {'SE':>10} {'z':>8} "
            f"{'p_raw':>10} {'p_fdr':>10} {'N':>8}")

    print(f"\n{sep}")
    print(title)
    print(sep2)
    print(hdr)
    print(sep2)

    fe = df[~df['is_re']] if 'is_re' in df.columns else df
    for _, row in fe.iterrows():
        p_fdr = row.get('p_fdr', np.nan)
        p_fdr_str = f"{p_fdr:>10.4f}" if np.isfinite(p_fdr) else f"{'—':>10}"
        stars = sig_stars(p_fdr) if np.isfinite(p_fdr) else ''
        print(f"  {row['term']:<43} {row['beta']:>10.5f} {row['se']:>10.5f} "
              f"{row['z']:>8.3f} {row['p_raw']:>10.4f} "
              f"{p_fdr_str} {int(row['nobs']):>8,}  {stars}")

    re = df[df['is_re']] if 'is_re' in df.columns else pd.DataFrame()
    if not re.empty:
        print(f"  {'— random effects —'}")
        for _, row in re.iterrows():
            beta_str = f"{row['beta']:>10.5f}" if np.isfinite(row['beta']) \
                       else f"{'—':>10}"
            print(f"  {row['term']:<43} {beta_str}")

    print(sep2)
    print("  Outcome = RSA_r (raw Pearson r, mixed power = frac+osc).  DBOY1 only.")
    print("  FDR: BH within fixed effects only.  "
          "† p<.10  * p<.05  ** p<.01  *** p<.001")
    print(f"  Reference: band=theta, hemisphere=LHP")
    print(sep)


# ============================================================================
# STEP 1
# ============================================================================

def step1(df: pd.DataFrame) -> List[pd.DataFrame]:
    print(f"\n{'#'*70}")
    print("STEP 1 — Does RSA_r track T_lag / SP_lag at all?")
    print(f"{'#'*70}")
    print("  Data: all bands pooled, both hemispheres")

    results = []
    for pred in ['T_lag', 'SP_lag']:
        res = fit_lmm(
            df,
            formula_cols = [pred, 'output_lag'],
            formula_rhs  = f'{pred} + output_lag',
            label        = f'Step1 {pred}',
        )
        rdf = result_to_df(res, step='Step1',
                           model_desc=f'RSA_r ~ {pred} + output_lag')
        print_table(rdf,
                    f"Step 1 | RSA_r ~ {pred} + output_lag  [all bands, combined]")
        results.append(rdf)

    return results


# ============================================================================
# STEP 2
# ============================================================================

def step2(df: pd.DataFrame) -> List[pd.DataFrame]:
    print(f"\n{'#'*70}")
    print("STEP 2 — Is the effect theta-specific?")
    print(f"{'#'*70}")
    print("  Data: all bands pooled, both hemispheres")
    print("  Reference band: theta")

    df = df.copy()
    for band in NON_REF_BANDS:
        df[f'band_{band}'] = (df['band'] == band).astype(float)

    results = []
    for pred in ['T_lag', 'SP_lag']:

        for band in NON_REF_BANDS:
            df[f'{pred}_x_{band}'] = df[pred] * df[f'band_{band}']

        band_cols  = [f'band_{b}'     for b in NON_REF_BANDS]
        inter_cols = [f'{pred}_x_{b}' for b in NON_REF_BANDS]

        formula_cols = [pred] + band_cols + inter_cols + ['output_lag']
        formula_rhs  = (f'{pred} + '
                        + ' + '.join(band_cols) + ' + '
                        + ' + '.join(inter_cols)
                        + ' + output_lag')

        res = fit_lmm(df, formula_cols, formula_rhs, label=f'Step2 {pred}')
        rdf = result_to_df(res, step='Step2',
                           model_desc=f'RSA_r ~ {pred} * band + output_lag')
        print_table(rdf,
                    f"Step 2 | RSA_r ~ {pred} * band + output_lag  "
                    f"[all bands, combined]\n"
                    f"  KEY: {pred}_x_{{band}} significant → slope ≠ theta")
        results.append(rdf)

    return results


# ============================================================================
# STEP 3
# ============================================================================

def step3(df: pd.DataFrame) -> List[pd.DataFrame]:
    print(f"\n{'#'*70}")
    print("STEP 3 — Is the theta effect lateralized?  (theta band only)")
    print(f"{'#'*70}")

    theta_df = df[df['band'] == 'theta'].copy()
    print(f"  Theta rows: {len(theta_df):,}  "
          f"subjects: {theta_df['subject'].nunique()}")

    results = []
    for pred in ['T_lag', 'SP_lag']:
        inter_col = f'{pred}_x_hemi'
        theta_df[inter_col] = theta_df[pred] * theta_df['hemisphere_dummy']

        # Combined: interaction test
        res = fit_lmm(
            theta_df,
            formula_cols = [pred, 'hemisphere_dummy', inter_col, 'output_lag'],
            formula_rhs  = (f'{pred} + hemisphere_dummy + {inter_col}'
                            f' + output_lag'),
            label        = f'Step3 {pred} combined',
        )
        rdf = result_to_df(res, step='Step3',
                           model_desc=f'RSA_r ~ {pred} * hemisphere + output_lag',
                           hemisphere='combined')
        print_table(rdf,
                    f"Step 3 | RSA_r ~ {pred} * hemisphere + output_lag  "
                    f"[theta, combined]\n"
                    f"  KEY TERM: {inter_col}  "
                    f"(significant → LHP slope ≠ RHP slope)")
        results.append(rdf)

        # Simple slopes per hemisphere
        for hemi, hemi_label in [('left', 'LHP'), ('right', 'RHP')]:
            hd = theta_df[theta_df['hemisphere'] == hemi].copy()
            res_h = fit_lmm(
                hd,
                formula_cols = [pred, 'output_lag'],
                formula_rhs  = f'{pred} + output_lag',
                label        = f'Step3 {pred} {hemi_label}',
            )
            rdf_h = result_to_df(res_h, step='Step3',
                                 model_desc=f'RSA_r ~ {pred} + output_lag',
                                 hemisphere=hemi_label)
            print_table(rdf_h,
                        f"Step 3 | RSA_r ~ {pred} + output_lag  "
                        f"[theta, {hemi_label} only]")
            results.append(rdf_h)

    return results


# ============================================================================
# STEP 4
# ============================================================================

def step4(df: pd.DataFrame) -> List[pd.DataFrame]:
    print(f"\n{'#'*70}")
    print("STEP 4 — Do T_lag and SP_lag survive mutual control?  (theta only)")
    print(f"{'#'*70}")

    theta_df = df[df['band'] == 'theta'].copy()
    print(f"  Theta rows: {len(theta_df):,}  "
          f"subjects: {theta_df['subject'].nunique()}")

    formula_cols = ['T_lag', 'SP_lag', 'output_lag']
    formula_rhs  = 'T_lag + SP_lag + output_lag'

    results = []
    for subset_label, subset_df in [
            ('combined', theta_df),
            ('LHP',      theta_df[theta_df['hemisphere'] == 'left']),
            ('RHP',      theta_df[theta_df['hemisphere'] == 'right'])]:

        res = fit_lmm(
            subset_df,
            formula_cols = formula_cols,
            formula_rhs  = formula_rhs,
            label        = f'Step4 {subset_label}',
        )
        rdf = result_to_df(res, step='Step4',
                           model_desc='RSA_r ~ T_lag + SP_lag + output_lag',
                           hemisphere=subset_label)
        print_table(rdf,
                    f"Step 4 | RSA_r ~ T_lag + SP_lag + output_lag  "
                    f"[theta, {subset_label}]")
        results.append(rdf)

    return results


# ============================================================================
# ENTRY POINT
# ============================================================================

def main():
    print(f"\n{'='*70}")
    print("RSA LAG — SEQUENTIAL LMM  |  DBOY1 only  |  RAW SCALE  |  MIXED (frac+osc)")
    print(f"{'='*70}")

    df = load_data()
    if df is None or df.empty:
        print("No data loaded. Exiting.")
        return

    all_results = []
    all_results += step1(df)
    all_results += step2(df)
    all_results += step3(df)
    all_results += step4(df)

    # Save CSV
    if all_results:
        master = pd.concat(all_results, ignore_index=True)
        csv_path = OUTPUT_DIR / 'SEQUENTIAL_LMM_RAW_DBOY1_mixed_results.csv'   # ← mixed
        master.to_csv(csv_path, index=False)
        print(f"\n  ✓ Results → {csv_path}  ({len(master):,} rows)")

        # Save text
        txt_path = OUTPUT_DIR / 'SEQUENTIAL_LMM_RAW_DBOY1_mixed_results.txt'   # ← mixed
        lines = []
        for rdf in all_results:
            if rdf.empty:
                continue
            lines.append(f"\n{'='*80}")
            lines.append(f"{rdf['step'].iloc[0]}  |  "
                         f"{rdf['model'].iloc[0]}  |  "
                         f"{rdf['hemisphere'].iloc[0]}")
            lines.append(f"{'='*80}")
            for _, row in rdf[~rdf['is_re']].iterrows():
                p_fdr = row['p_fdr']
                stars = sig_stars(p_fdr) if np.isfinite(p_fdr) else ''
                p_fdr_s = f"{p_fdr:.4f}" if np.isfinite(p_fdr) else '—'
                lines.append(
                    f"  {row['term']:<40} β={row['beta']:>10.5f}  "
                    f"SE={row['se']:>9.5f}  z={row['z']:>7.3f}  "
                    f"p={row['p_raw']:>8.4f}  p_fdr={p_fdr_s:>8}  {stars}"
                )
        with open(txt_path, 'w') as f:
            f.write('\n'.join(lines))
        print(f"  ✓ Text    → {txt_path}")

    print(f"\n{'='*70}")
    print("INTERPRETATION GUIDE")
    print(f"{'='*70}")
    print("""
  Step 1: T_lag/SP_lag significant controlling for output_lag?
          → genuine clustering effect, not recall-order artifact.

  Step 2: predictor_x_alpha/beta/gamma significant?
          → theta slope differs from that band → theta specificity confirmed.
          NOT significant → effect is broadband, not theta-specific.

  Step 3: predictor_x_hemi significant?
          → LHP and RHP slopes are genuinely different.
          Simple slopes show which hemisphere drives the effect.

  Step 4: Both T_lag and SP_lag survive together + output_lag?
          → temporal and spatial clustering are INDEPENDENT.

  NOTE: RSA vectors here used mixed power (frac + osc per freq × channel),
        capturing the full broadband power profile rather than fractal or
        oscillatory components in isolation.
    """)


if __name__ == '__main__':
    main()


RSA LAG — SEQUENTIAL LMM  |  DBOY1 only  |  RAW SCALE  |  MIXED (frac+osc)
  Loaded ALL_SUBJECTS_DBOY1_hc_allbands_rsa_lag_mixed.csv  (13,028 rows)

  Total rows : 13,028
  Subjects   : 34
  Sessions   : 6
  Bands      : ['alpha', 'beta', 'gamma', 'theta']
  Regions    : ['LHP', 'RHP']

  Predictor scales (raw):
    RSA_r        mean=0.195  sd=0.399  range=[-1.00, 1.00]
    T_lag        mean=4.523  sd=2.914  range=[1.00, 11.00]
    SP_lag       mean=70.149  sd=30.696  range=[12.96, 142.72]
    output_lag   mean=2.450  sd=1.560  range=[1.00, 10.00]

######################################################################
STEP 1 — Does RSA_r track T_lag / SP_lag at all?
######################################################################
  Data: all bands pooled, both hemispheres

  [Step1 T_lag] RSA_r ~ T_lag + output_lag
  [Step1 T_lag] N=13,028  subjects=34
  [Step1 T_lag] nested RE  optimizer=nm  converged=True

Step 1 | RSA_r ~ T_lag + output_lag  [all bands, combined]
------------

In [5]:
#!/usr/bin/env python3
"""
RSA Lag — Sequential LMM Analysis (4 Steps) — RAW SCALE, DBOY1 only
========================================================================
*** MIXED POWER variant (frac + osc) — reads from rsa_lag_hc_bands_mixed/ ***

Representational vectors were built as:
    mixed[freq, ch] = ret_frac_f{freq} + ret_osc_f{freq}
then sliced to band frequency indices (freq × channel, flattened).

Betas are in original units:
  RSA_r         : Pearson r
  T_lag         : number of list positions separating the two items
  SP_lag        : Euclidean distance in store-coordinate units
  semantic_sim  : Word2Vec cosine similarity between recalled words
  output_lag    : number of recall output positions between the two items

Predictors tested in all steps: T_lag, SP_lag, semantic_sim

Steps
-----
  Step 1 — RSA_r ~ predictor + output_lag + (1|subj)
            All bands pooled, both hemispheres.
            Run separately for each of: T_lag, SP_lag, semantic_sim.

  Step 2 — RSA_r ~ predictor * band (explicit dummies, theta = reference)
                 + output_lag + (1|subj)
            Tests whether predictor slope differs across bands.
            Run separately for each predictor.

  Step 3 — RSA_r ~ predictor * hemisphere + output_lag + (1|subj)
            Theta band only.  Tests LHP vs RHP asymmetry.
            Also run separately for LHP and RHP.
            Run for each predictor.

  Step 4 — RSA_r ~ T_lag + SP_lag + semantic_sim + output_lag + (1|subj)
            Theta band only.  Tests independence of all three predictors
            simultaneously.

Input  : ./rsa_lag_hc_bands_mixed/ALL_SUBJECTS_DBOY1_hc_allbands_rsa_lag_mixed.csv
Output : ./rsa_lag_hc_bands_mixed/SEQUENTIAL_LMM_RAW_DBOY1_mixed_results.csv
         ./rsa_lag_hc_bands_mixed/SEQUENTIAL_LMM_RAW_DBOY1_mixed_results.txt
"""

import warnings
from pathlib import Path
from typing import List, Optional

import numpy as np
import pandas as pd
from statsmodels.regression.mixed_linear_model import MixedLM
from statsmodels.stats.multitest import fdrcorrection

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

INPUT_DIR  = Path('./rsa_lag_hc_bands_mixed')
OUTPUT_DIR = Path('./rsa_lag_hc_bands_mixed')

BANDS         = ['theta', 'alpha', 'beta', 'gamma']
NON_REF_BANDS = ['alpha', 'beta', 'gamma']   # theta = reference

# All three predictors treated symmetrically across Steps 1–3
PREDICTORS = ['T_lag', 'SP_lag', 'semantic_sim']

OUTCOME = 'RSA_r'


# ============================================================================
# DATA LOADING
# ============================================================================

def load_data() -> Optional[pd.DataFrame]:
    fpath = INPUT_DIR / "ALL_SUBJECTS_DBOY1_hc_allbands_rsa_lag_mixed.csv"
    if not fpath.exists():
        print(f"  ✗ File not found: {fpath}")
        return None

    df = pd.read_csv(fpath)
    print(f"  Loaded {fpath.name}  ({len(df):,} rows)")

    if 'hemisphere' not in df.columns:
        df['hemisphere'] = df['region'].map({'LHP': 'left', 'RHP': 'right'})

    df['hemisphere_dummy'] = (df['hemisphere'] == 'right').astype(int)

    # Report semantic_sim coverage
    n_total  = len(df)
    n_semsim = df['semantic_sim'].notna().sum()
    print(f"\n  Total rows       : {n_total:,}")
    print(f"  Subjects         : {df['subject'].nunique()}")
    print(f"  Sessions         : {df['session'].nunique()}")
    print(f"  Bands            : {sorted(df['band'].unique().tolist())}")
    print(f"  Regions          : {sorted(df['region'].unique().tolist())}")
    print(f"  semantic_sim     : {n_semsim:,} non-NaN "
          f"({100*n_semsim/n_total:.1f}% coverage)")

    print(f"\n  Predictor scales (raw):")
    for col in ['RSA_r', 'T_lag', 'SP_lag', 'semantic_sim', 'output_lag']:
        if col in df.columns:
            s = df[col].dropna()
            print(f"    {col:<14} mean={s.mean():.3f}  "
                  f"sd={s.std():.3f}  "
                  f"range=[{s.min():.3f}, {s.max():.3f}]  "
                  f"N={len(s):,}")

    return df


# ============================================================================
# LMM FITTING
# ============================================================================

def fit_lmm(df: pd.DataFrame,
            formula_cols: List[str],
            formula_rhs: str,
            label: str) -> Optional[object]:
    """
    Fit RSA_r ~ formula_rhs.

    Random effects strategy (tried in order):
      1. Nested: (1|subject) + (1|subject:session)  via vc_formula
      2. Simple : (1|subject) only — fallback if nested hits a boundary
    """
    df = df.copy()
    df['subj_sess'] = df['subject'].astype(str) + '_' + df['session'].astype(str)

    keep = [OUTCOME] + [c for c in formula_cols if c in df.columns] \
           + ['subject', 'subj_sess']
    df   = df[keep].dropna()

    n = len(df)
    if n < 20:
        print(f"  [{label}] Too few rows ({n}) — skipping")
        return None

    formula = f"{OUTCOME} ~ {formula_rhs}"
    print(f"\n  [{label}] {formula}")
    print(f"  [{label}] N={n:,}  subjects={df['subject'].nunique()}")

    def _is_boundary(res) -> bool:
        if res is None or not np.isfinite(res.llf):
            return True
        for k in res.params.index:
            if 'subj_sess' in k.lower():
                if abs(res.params[k] - 16.0) < 1e-6:
                    return True
                if not np.isfinite(res.bse[k]):
                    return True
        return False

    # Attempt 1: nested random effects
    result = None
    for method in ['lbfgs', 'nm', 'powell']:
        try:
            model = MixedLM.from_formula(
                formula,
                data       = df,
                groups     = df['subject'],
                vc_formula = {'subj_sess': '0 + C(subj_sess)'},
            )
            res = model.fit(reml=True, method=method)
            if np.isfinite(res.llf) and not _is_boundary(res):
                print(f"  [{label}] nested RE  optimizer={method}  "
                      f"converged={getattr(res, 'converged', None)}")
                result = res
                break
        except Exception:
            pass

    # Attempt 2: subject-only random effects
    if result is None or _is_boundary(result):
        print(f"  [{label}] ⚠ Nested RE boundary — "
              f"falling back to (1|subject) only")
        for method in ['lbfgs', 'nm', 'powell']:
            try:
                model = MixedLM.from_formula(
                    formula,
                    data   = df,
                    groups = df['subject'],
                )
                res = model.fit(reml=True, method=method)
                if np.isfinite(res.llf):
                    print(f"  [{label}] subject-only RE  optimizer={method}  "
                          f"converged={getattr(res, 'converged', None)}")
                    result = res
                    break
            except Exception as e:
                print(f"  [{label}] subject-only {method} failed: {e}")
                result = None

    if result is None:
        print(f"  [{label}] WARNING: all fits unsuccessful")
    return result


# ============================================================================
# RESULT FORMATTING
# ============================================================================

def sig_stars(p: float) -> str:
    return ('***' if p < 0.001 else '**' if p < 0.01 else
            '*'   if p < 0.05  else '†'  if p < 0.10  else '')


def result_to_df(result, step: str, model_desc: str,
                 hemisphere: str = 'combined',
                 predictor: str = '') -> pd.DataFrame:
    if result is None:
        return pd.DataFrame()

    rows = []
    for term in result.params.index:
        p_val = result.pvalues[term]
        is_re = (not np.isfinite(result.bse[term]) or
                 not np.isfinite(p_val) or
                 'var' in term.lower())
        rows.append({
            'step':       step,
            'predictor':  predictor,
            'model':      model_desc,
            'hemisphere': hemisphere,
            'term':       term,
            'beta':       result.params[term],
            'se':         result.bse[term],
            'z':          result.tvalues[term],
            'p_raw':      p_val,
            'p_fdr':      np.nan,
            'is_re':      is_re,
            'nobs':       int(result.nobs),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    fe_mask = (~df['is_re']) & df['p_raw'].notna() & np.isfinite(df['p_raw'])
    if fe_mask.sum() > 0:
        _, fdr_vals = fdrcorrection(df.loc[fe_mask, 'p_raw'].values)
        df.loc[fe_mask, 'p_fdr'] = fdr_vals

    return df


def print_table(df: pd.DataFrame, title: str):
    if df.empty:
        print(f"\n  [no results for: {title}]")
        return

    sep  = '=' * 95
    sep2 = '-' * 95
    hdr  = (f"{'Term':<45} {'β':>10} {'SE':>10} {'z':>8} "
            f"{'p_raw':>10} {'p_fdr':>10} {'N':>8}")

    print(f"\n{sep}")
    print(title)
    print(sep2)
    print(hdr)
    print(sep2)

    fe = df[~df['is_re']] if 'is_re' in df.columns else df
    for _, row in fe.iterrows():
        p_fdr = row.get('p_fdr', np.nan)
        p_fdr_str = f"{p_fdr:>10.4f}" if np.isfinite(p_fdr) else f"{'—':>10}"
        stars = sig_stars(p_fdr) if np.isfinite(p_fdr) else ''
        print(f"  {row['term']:<43} {row['beta']:>10.5f} {row['se']:>10.5f} "
              f"{row['z']:>8.3f} {row['p_raw']:>10.4f} "
              f"{p_fdr_str} {int(row['nobs']):>8,}  {stars}")

    re = df[df['is_re']] if 'is_re' in df.columns else pd.DataFrame()
    if not re.empty:
        print(f"  {'— random effects —'}")
        for _, row in re.iterrows():
            beta_str = f"{row['beta']:>10.5f}" if np.isfinite(row['beta']) \
                       else f"{'—':>10}"
            print(f"  {row['term']:<43} {beta_str}")

    print(sep2)
    print("  Outcome = RSA_r (raw Pearson r, mixed power = frac+osc).  DBOY1 only.")
    print("  FDR: BH within fixed effects only.  "
          "† p<.10  * p<.05  ** p<.01  *** p<.001")
    print(f"  Reference: band=theta, hemisphere=LHP")
    print(sep)


# ============================================================================
# STEP 1 — univariate effects (one predictor at a time)
# ============================================================================

def step1(df: pd.DataFrame) -> List[pd.DataFrame]:
    print(f"\n{'#'*70}")
    print("STEP 1 — Does RSA_r track each predictor?")
    print(f"{'#'*70}")
    print("  Data: all bands pooled, both hemispheres")
    print(f"  Predictors: {PREDICTORS}")

    results = []
    for pred in PREDICTORS:
        res = fit_lmm(
            df,
            formula_cols = [pred, 'output_lag'],
            formula_rhs  = f'{pred} + output_lag',
            label        = f'Step1 {pred}',
        )
        rdf = result_to_df(res, step='Step1',
                           model_desc=f'RSA_r ~ {pred} + output_lag',
                           predictor=pred)
        print_table(rdf,
                    f"Step 1 | RSA_r ~ {pred} + output_lag  [all bands, combined]")
        results.append(rdf)

    return results


# ============================================================================
# STEP 2 — band specificity (predictor × band interaction)
# ============================================================================

def step2(df: pd.DataFrame) -> List[pd.DataFrame]:
    print(f"\n{'#'*70}")
    print("STEP 2 — Is each effect theta-specific?")
    print(f"{'#'*70}")
    print("  Data: all bands pooled, both hemispheres")
    print("  Reference band: theta")

    df = df.copy()
    for band in NON_REF_BANDS:
        df[f'band_{band}'] = (df['band'] == band).astype(float)

    results = []
    for pred in PREDICTORS:

        for band in NON_REF_BANDS:
            df[f'{pred}_x_{band}'] = df[pred] * df[f'band_{band}']

        band_cols  = [f'band_{b}'     for b in NON_REF_BANDS]
        inter_cols = [f'{pred}_x_{b}' for b in NON_REF_BANDS]

        formula_cols = [pred] + band_cols + inter_cols + ['output_lag']
        formula_rhs  = (f'{pred} + '
                        + ' + '.join(band_cols) + ' + '
                        + ' + '.join(inter_cols)
                        + ' + output_lag')

        res = fit_lmm(df, formula_cols, formula_rhs, label=f'Step2 {pred}')
        rdf = result_to_df(res, step='Step2',
                           model_desc=f'RSA_r ~ {pred} * band + output_lag',
                           predictor=pred)
        print_table(rdf,
                    f"Step 2 | RSA_r ~ {pred} * band + output_lag  "
                    f"[all bands, combined]\n"
                    f"  KEY: {pred}_x_{{band}} significant → slope ≠ theta")
        results.append(rdf)

    return results


# ============================================================================
# STEP 3 — hemispheric asymmetry (theta band only)
# ============================================================================

def step3(df: pd.DataFrame) -> List[pd.DataFrame]:
    print(f"\n{'#'*70}")
    print("STEP 3 — Is each theta effect lateralized?  (theta band only)")
    print(f"{'#'*70}")

    theta_df = df[df['band'] == 'theta'].copy()
    print(f"  Theta rows: {len(theta_df):,}  "
          f"subjects: {theta_df['subject'].nunique()}")

    results = []
    for pred in PREDICTORS:
        inter_col = f'{pred}_x_hemi'
        theta_df[inter_col] = theta_df[pred] * theta_df['hemisphere_dummy']

        # Combined: interaction test
        res = fit_lmm(
            theta_df,
            formula_cols = [pred, 'hemisphere_dummy', inter_col, 'output_lag'],
            formula_rhs  = (f'{pred} + hemisphere_dummy + {inter_col}'
                            f' + output_lag'),
            label        = f'Step3 {pred} combined',
        )
        rdf = result_to_df(res, step='Step3',
                           model_desc=f'RSA_r ~ {pred} * hemisphere + output_lag',
                           hemisphere='combined',
                           predictor=pred)
        print_table(rdf,
                    f"Step 3 | RSA_r ~ {pred} * hemisphere + output_lag  "
                    f"[theta, combined]\n"
                    f"  KEY TERM: {inter_col}  "
                    f"(significant → LHP slope ≠ RHP slope)")
        results.append(rdf)

        # Simple slopes per hemisphere
        for hemi, hemi_label in [('left', 'LHP'), ('right', 'RHP')]:
            hd = theta_df[theta_df['hemisphere'] == hemi].copy()
            res_h = fit_lmm(
                hd,
                formula_cols = [pred, 'output_lag'],
                formula_rhs  = f'{pred} + output_lag',
                label        = f'Step3 {pred} {hemi_label}',
            )
            rdf_h = result_to_df(res_h, step='Step3',
                                 model_desc=f'RSA_r ~ {pred} + output_lag',
                                 hemisphere=hemi_label,
                                 predictor=pred)
            print_table(rdf_h,
                        f"Step 3 | RSA_r ~ {pred} + output_lag  "
                        f"[theta, {hemi_label} only]")
            results.append(rdf_h)

    return results


# ============================================================================
# STEP 4 — mutual control of all three predictors (theta only)
# ============================================================================

def step4(df: pd.DataFrame) -> List[pd.DataFrame]:
    print(f"\n{'#'*70}")
    print("STEP 4 — Do T_lag, SP_lag, and semantic_sim survive mutual control?")
    print("         (theta band only)")
    print(f"{'#'*70}")

    theta_df = df[df['band'] == 'theta'].copy()
    print(f"  Theta rows (before semsim dropna): {len(theta_df):,}  "
          f"subjects: {theta_df['subject'].nunique()}")

    # Report complete-case count
    complete = theta_df[['T_lag', 'SP_lag', 'semantic_sim',
                          'output_lag', OUTCOME]].dropna()
    print(f"  Theta rows (complete cases):        {len(complete):,}")

    formula_cols = ['T_lag', 'SP_lag', 'semantic_sim', 'output_lag']
    formula_rhs  = 'T_lag + SP_lag + semantic_sim + output_lag'

    results = []
    for subset_label, subset_df in [
            ('combined', theta_df),
            ('LHP',      theta_df[theta_df['hemisphere'] == 'left']),
            ('RHP',      theta_df[theta_df['hemisphere'] == 'right'])]:

        res = fit_lmm(
            subset_df,
            formula_cols = formula_cols,
            formula_rhs  = formula_rhs,
            label        = f'Step4 {subset_label}',
        )
        rdf = result_to_df(
            res, step='Step4',
            model_desc='RSA_r ~ T_lag + SP_lag + semantic_sim + output_lag',
            hemisphere=subset_label,
            predictor='all',
        )
        print_table(rdf,
                    f"Step 4 | RSA_r ~ T_lag + SP_lag + semantic_sim + output_lag"
                    f"  [theta, {subset_label}]")
        results.append(rdf)

    return results


# ============================================================================
# ENTRY POINT
# ============================================================================

def main():
    print(f"\n{'='*70}")
    print("RSA LAG — SEQUENTIAL LMM  |  DBOY1 only  |  RAW SCALE  |  MIXED (frac+osc)")
    print(f"Predictors: T_lag, SP_lag, semantic_sim  (all treated in parallel)")
    print(f"{'='*70}")

    df = load_data()
    if df is None or df.empty:
        print("No data loaded. Exiting.")
        return

    all_results = []
    all_results += step1(df)
    all_results += step2(df)
    all_results += step3(df)
    all_results += step4(df)

    # Save CSV
    if all_results:
        master = pd.concat(all_results, ignore_index=True)
        csv_path = OUTPUT_DIR / 'SEQUENTIAL_LMM_RAW_DBOY1_mixed_results.csv'
        master.to_csv(csv_path, index=False)
        print(f"\n  ✓ Results → {csv_path}  ({len(master):,} rows)")

        # Save text
        txt_path = OUTPUT_DIR / 'SEQUENTIAL_LMM_RAW_DBOY1_mixed_results.txt'
        lines = []
        for rdf in all_results:
            if rdf.empty:
                continue
            lines.append(f"\n{'='*80}")
            lines.append(f"{rdf['step'].iloc[0]}  |  "
                         f"pred={rdf['predictor'].iloc[0]}  |  "
                         f"{rdf['model'].iloc[0]}  |  "
                         f"{rdf['hemisphere'].iloc[0]}")
            lines.append(f"{'='*80}")
            for _, row in rdf[~rdf['is_re']].iterrows():
                p_fdr = row['p_fdr']
                stars = sig_stars(p_fdr) if np.isfinite(p_fdr) else ''
                p_fdr_s = f"{p_fdr:.4f}" if np.isfinite(p_fdr) else '—'
                lines.append(
                    f"  {row['term']:<40} β={row['beta']:>10.5f}  "
                    f"SE={row['se']:>9.5f}  z={row['z']:>7.3f}  "
                    f"p={row['p_raw']:>8.4f}  p_fdr={p_fdr_s:>8}  {stars}"
                )
        with open(txt_path, 'w') as f:
            f.write('\n'.join(lines))
        print(f"  ✓ Text    → {txt_path}")

    print(f"\n{'='*70}")
    print("INTERPRETATION GUIDE")
    print(f"{'='*70}")
    print("""
  Step 1: Each predictor tested separately controlling for output_lag.
          T_lag/SP_lag significant → genuine spatial/temporal clustering.
          semantic_sim significant → neural similarity tracks word meaning.

  Step 2: predictor_x_alpha/beta/gamma significant?
          → theta slope differs from that band → theta specificity confirmed.
          NOT significant → effect is broadband.

  Step 3: predictor_x_hemi significant?
          → LHP and RHP slopes differ.
          Simple slopes reveal which hemisphere drives the effect.

  Step 4: All three predictors (T_lag, SP_lag, semantic_sim) + output_lag
          entered simultaneously in theta band.
          Surviving effects are independent of the other two dimensions.
          Note: rows with NaN semantic_sim are dropped from Step 4 models.

  NOTE: RSA vectors used mixed power (frac + osc per freq × channel).
    """)


if __name__ == '__main__':
    main()


RSA LAG — SEQUENTIAL LMM  |  DBOY1 only  |  RAW SCALE  |  MIXED (frac+osc)
Predictors: T_lag, SP_lag, semantic_sim  (all treated in parallel)
  Loaded ALL_SUBJECTS_DBOY1_hc_allbands_rsa_lag_mixed.csv  (13,028 rows)

  Total rows       : 13,028
  Subjects         : 34
  Sessions         : 6
  Bands            : ['alpha', 'beta', 'gamma', 'theta']
  Regions          : ['LHP', 'RHP']
  semantic_sim     : 9,960 non-NaN (76.5% coverage)

  Predictor scales (raw):
    RSA_r          mean=0.195  sd=0.399  range=[-1.000, 1.000]  N=13,028
    T_lag          mean=4.523  sd=2.914  range=[1.000, 11.000]  N=13,028
    SP_lag         mean=70.149  sd=30.696  range=[12.962, 142.722]  N=13,028
    semantic_sim   mean=0.278  sd=0.176  range=[-0.059, 0.821]  N=9,960
    output_lag     mean=2.450  sd=1.560  range=[1.000, 10.000]  N=13,028

######################################################################
STEP 1 — Does RSA_r track each predictor?
######################################################